In [1]:
import os
import sys

import torch
import torch.optim as optim
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

import re
import argparse
import numpy as np
from tqdm import tqdm 
from PIL import Image
from einops import rearrange
from natsort import natsorted
from omegaconf import OmegaConf

sys.path.append("..")
import utils
from diffusion import create_diffusion    
from models import get_models
from diffusers.models import AutoencoderKL
from models.clip import TextEmbedder
from datasets import video_transforms
from utils import mask_generation_before
from diffusers.utils.import_utils import is_xformers_available

/home/brina/miniconda3/envs/seine/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
config_file = os.path.join(base_dir, "configs/train.yaml")

In [3]:
# Custom Dataset class
class VideoDataset(Dataset):
    def __init__(self, data_dir, transform, num_frames=32, mask_type="onelast1"):
        self.data_dir = data_dir
        self.transform = transform
        self.num_frames = num_frames
        self.mask_type = mask_type
        self.video_paths = self._get_video_paths()

    def _get_video_paths(self):
        video_paths = []
        # Iterate through all subdirectories (classes) in data_dir
        for class_dir in os.listdir(self.data_dir):
            class_path = os.path.join(self.data_dir, class_dir)
            if os.path.isdir(class_path):
                # Iterate through each video folder in the class
                for video_dir in os.listdir(class_path):
                    video_path = os.path.join(class_path, video_dir)
                    if os.path.isdir(video_path):
                        video_paths.append((video_path, class_dir))  # Store class along with path
        print(f"Total videos: {len(video_paths)}")
        return video_paths

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        input_path, class_name = self.video_paths[idx]
        video_frames, n = self._load_video(input_path)
        return video_frames, class_name  # Returning video frames and class name

    def _load_video(self, input_path):
        file_list = os.listdir(input_path)
        video_frames = []
        
        if self.mask_type.startswith('onelast'):
            num = int(self.mask_type.split('onelast')[-1])
            # Get first and last frame
            first_frame_path = os.path.join(input_path, natsorted(file_list)[0])
            last_frame_path = os.path.join(input_path, natsorted(file_list)[-1])
            first_frame = torch.as_tensor(np.array(Image.open(first_frame_path), dtype=np.uint8, copy=True)).unsqueeze(0)
            last_frame = torch.as_tensor(np.array(Image.open(last_frame_path), dtype=np.uint8, copy=True)).unsqueeze(0)
            
            for _ in range(num):
                video_frames.append(first_frame)
            
            # Add zeros to frames
            num_zeros = self.num_frames - 2 * num
            for _ in range(num_zeros):
                zeros = torch.zeros_like(first_frame)
                video_frames.append(zeros)
            
            for _ in range(num):
                video_frames.append(last_frame)
            video_frames = torch.cat(video_frames, dim=0).permute(0, 3, 1, 2)  # f,c,h,w
            video_frames = self.transform(video_frames)
        else:
            for file in file_list:
                if file.endswith('jpg') or file.endswith('png'):
                    image = torch.as_tensor(np.array(Image.open(os.path.join(input_path, file)), dtype=np.uint8, copy=True)).unsqueeze(0)
                    video_frames.append(image)
            video_frames = torch.cat(video_frames, dim=0).permute(0, 3, 1, 2)  # f,c,h,w
            video_frames = self.transform(video_frames)

        return video_frames, 0  # Returning 0 for n, as it seems like it's unused

In [4]:
def collate_fn(batch):
    """
    Custom collate function to handle video frames and class names separately.
    """
    video_frames, class_names = zip(*batch)  # Unpack batch tuples
    video_frames = torch.stack(video_frames, dim=0)  # Stack tensors
    class_names = list(class_names)  # Convert tuple to list
    return video_frames, class_names  # Keep class names as a list

def camel_to_snake(text):
    return re.sub(r'(?<!^)(?=[A-Z])', ' ', text).lower()

In [ ]:
def main(args):
    # Setup PyTorch:
    if args.seed:
        torch.manual_seed(args.seed)
    torch.set_grad_enabled(True)  # Enable gradients for training
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load model:
    latent_h = args.image_size[0] // 8
    latent_w = args.image_size[1] // 8
    args.image_h = args.image_size[0]
    args.image_w = args.image_size[1]
    args.latent_h = latent_h
    args.latent_w = latent_w
    
    if args.ckpt != "":
        # Load model if checkpoint is provided for resuming training
        print('loading model from checkpoint...')
        model = get_models(args).to(device)
        state_dict = torch.load(args.ckpt, map_location=device)['ema']
        model.load_state_dict(state_dict)
        print('loading succeed')
    else:
        # Initialize the model
        print('Initializing model...')
        model = get_models(args).to(device)

    # Model preparation (e.g., enabling memory-efficient attention if required)
    if args.enable_xformers_memory_efficient_attention:
        if is_xformers_available():
            model.enable_xformers_memory_efficient_attention()
        else:
            raise ValueError("xformers is not available. Make sure it is installed correctly")

    # Set up optimizers
    optimizer = optim.AdamW(model.parameters(), lr=args.lr)
    criterion = nn.MSELoss()  # Use built-in Mean Squared Error loss

    # Load the diffusion model and VAE
    diffusion = create_diffusion(str(args.num_sampling_steps))
    vae = AutoencoderKL.from_pretrained(args.pretrained_model_path, subfolder="vae").to(device)
    text_encoder = TextEmbedder(args.pretrained_model_path).to(device)
    
    if args.use_fp16:
        print('Warning: Using half precision for training!')
        vae.to(dtype=torch.float16)
        model.to(dtype=torch.float16)
        text_encoder.to(dtype=torch.float16)

    # Define transformation
    transform_video = transforms.Compose([
        video_transforms.ToTensorVideo(),  # TCHW
        video_transforms.ResizeVideo((args.image_h, args.image_w)),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], inplace=True)
    ])

    # Prepare dataset
    data_dir = os.path.join(base_dir, args.data_path)
    video_dataset = VideoDataset(data_dir=data_dir, transform=transform_video, num_frames=args.num_frames, mask_type=args.mask_type)
    train_loader = DataLoader(video_dataset, batch_size=args.batch_size, shuffle=True, num_workers=8, collate_fn=collate_fn)

    # Training loop
    model.train()
    for epoch in range(args.num_epochs):
        total_loss = 0
        for batch_idx, (video_input, class_names) in tqdm(enumerate(train_loader), total=len(train_loader)):
            video_input = video_input.to(dtype=torch.float16, device=device)
            b, f, c, h, w = video_input.shape
            
            # Generate mask video
            mask = mask_generation_before(args.mask_type, video_input.shape, video_input.dtype, device)
            masked_video = video_input * (mask == 0)

            # Prepare latent space
            if args.use_fp16:
                z = torch.randn(1, 4, args.num_frames, args.latent_h, args.latent_w, dtype=torch.float16, device=device) # b,c,f,h,w
                masked_video = masked_video.to(dtype=torch.float16)
                mask = mask.to(dtype=torch.float16)
            else:
                z = torch.randn(1, 4, args.num_frames, args.latent_h, args.latent_w, device=device) # b,c,f,h,w

            # Convert video to latent space
            masked_video = rearrange(masked_video, 'b f c h w -> (b f) c h w')
            masked_video = vae.encode(masked_video).latent_dist.sample().mul_(0.18215)
            masked_video = rearrange(masked_video, '(b f) c h w -> b f c h w', b=b)
            mask = torch.nn.functional.interpolate(mask[:, :, 0, :], size=(args.latent_h, args.latent_w)).unsqueeze(1)

            # Generate text embedding
            prompt = args.text_prompt
            if prompt == []:
                if isinstance(class_names, list):
                    prompt = ' '.join(camel_to_snake(name) for name in class_names)
                else:
                    prompt = camel_to_snake(class_names)
                # prompt = args.input_path.split('/')[-1].split('.')[0].replace('_', ' ')
            else:
                prompt = prompt[0]
            prompt_base = prompt.replace(' ','_')
            prompt = prompt + args.additional_prompt
            print(prompt)

            # Handle classifier-free guidance (CFG)
            if args.do_classifier_free_guidance:
                masked_video = torch.cat([masked_video] * 2)
                mask = torch.cat([mask] * 2)
                z = torch.cat([z] * 2)
                prompt_all = [prompt] + [args.negative_prompt]
            else:
                masked_video = masked_video
                mask = mask
                z = z
                prompt_all = [prompt]

            # Encode text prompts
            text_prompt = text_encoder(text_prompts=prompt_all, train=False)
            model_kwargs = dict(encoder_hidden_states=text_prompt, 
                                    class_labels=None, 
                                    cfg_scale=args.cfg_scale,
                                    use_fp16=args.use_fp16)

            # Random timestep
            t = torch.randint(0, diffusion.num_timesteps, (video_input.shape[0],), device=device)
    
            # Compute loss
            with autocast(dtype=torch.float16):
                loss_terms = diffusion.training_losses(model.forward_with_cfg, masked_video, t,  model_kwargs=model_kwargs, noise=z, use_mask=args.use_mask)
                loss = loss_terms["loss"].mean()
            total_loss += loss.item()

            # Backpropagation and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Print loss for monitoring
            if batch_idx % 10 == 0:
                print(f'Epoch [{epoch+1}/{args.num_epochs}], Step [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

        # Save model checkpoint every epoch
        print(f'Epoch [{epoch+1}/{args.num_epochs}], Average Loss: {total_loss/len(train_loader):.4f}')
        save_checkpoint(model, optimizer, epoch, args.save_path)  # Save model checkpoint

    # Final model saving after training
    print('Training complete. Saving final model...')
    save_checkpoint(model, optimizer, args.num_epochs, args.save_path)

def save_checkpoint(model, optimizer, epoch, save_path):
    checkpoint_path = os.path.join(save_path, f'model_epoch_{epoch}.pth')
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, checkpoint_path)
    print(f'Saved checkpoint to {checkpoint_path}')

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", type=str, default=config_file)
    # args = parser.parse_args()
    args, unknown = parser.parse_known_args()
    omega_conf = OmegaConf.load(args.config)
    main(omega_conf)

Initializing model...
